In [14]:
import torch
import torchvision
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
from torch import nn  # All neural network modules
from torch.utils.data import DataLoader
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2
model = torchvision.models.mobilenet_v2(weights='DEFAULT')
model.avgpool = nn.Identity()
model.classifier = nn.Sequential( nn.Linear(1280,100), nn.ReLU(), nn.Linear(100,50), nn.ReLU(), nn.Linear(50,num_classes))
model.to(device)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [35]:
model.last_channel

1280

### Hyperparameters

In [42]:
in_channels = 3
learning_rate = 3e-4
batch_size = 16
num_epochs = 3


#### Load the Data

In [43]:
trans_train = transforms.Compose([ transforms.Resize(150),
                                  transforms.RandomRotation(30),
                                  transforms.RandomResizedCrop(32),
                                  transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor()])
trans_valid = transforms.Compose([ transforms.Resize(150),
                                  transforms.CenterCrop(32),
                                  transforms.ToTensor()
                                  ])

data_dir = "D:\Machine Learning\Datasets\Datasets\CatsVsDogs"
train_dataset = datasets.ImageFolder(data_dir + '/train', transform=trans_train)
test_dataset = datasets.ImageFolder(data_dir + '/val', transform=trans_valid)

batch_size = 1
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle= True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle =False)
type(train_dataset)

torchvision.datasets.folder.ImageFolder

In [44]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [45]:
for epoch in range(num_epochs):
    model.train()  # Important: set model to training mode (enables dropout, batchnorm updates, etc.)
    loop = tqdm(train_loader, leave=True)
    
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=device)
        targets = targets.to(device=device)  # targets should be torch.long (class indices: 0 or 1)
        
        # NO need for one-hot encoding with CrossEntropyLoss
        
        scores = model(data)          # shape: (batch_size, num_classes)
        loss = criterion(scores, targets)  # criterion expects (logits, long targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Update progress bar
        loop.set_description(f"Epoch [{epoch + 1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

  0%|          | 0/275 [00:00<?, ?it/s]


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 576, 1, 1])

In [ ]:
import torch
import torchvision
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
from torch import nn  # All neural network modules
from torch.utils.data import DataLoader
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt
import torch.nn.functional as F

## Load Pretrain model & Modify it 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
model = torchvision.models.vgg16(weights='DEFAULT')

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\dbda.STUDENTSDC/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|██████████| 528M/528M [00:09<00:00, 55.4MB/s] 


### Hyperparameters

In [ ]:
num_classes = 10
learning_rate = 1e-3
batch_size = 64
num_epochs = 3

In [ ]:
for param in model.parameters():
    param.requires_grad = False
    
model.avgpool = nn.Identity()
model.classifier = nn.Sequential( nn.Linear(512,100), nn.ReLU(), nn.Linear(100,50), nn.ReLU(), nn.Linear(50,num_classes))
model.to(device)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

### Load Data

In [ ]:
train_dataset = datasets.CIFAR10(
    root="dataset/", train=True, transform=transforms.ToTensor(), download=True
)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

100%|██████████| 170M/170M [00:35<00:00, 4.80MB/s] 


#### Loss and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

#### Train Network

In [ ]:
for epoch in range(num_epochs):
    losses = []

    for batch_idx, (data, targets) in enumerate(tqdm(train_loader)):
        data = data.to(device=device)
        targets = targets.to(device=device)
        scores = model(data)
        loss = criterion(scores,targets)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Cost at epoch {epoch} is {sum(losses) / len(losses) :.5f}")

100%|██████████| 782/782 [03:04<00:00,  4.24it/s]


Cost at epoch 0 is 1.27053


100%|██████████| 782/782 [02:43<00:00,  4.78it/s]


Cost at epoch 1 is 1.10030


100%|██████████| 782/782 [03:00<00:00,  4.34it/s]

Cost at epoch 2 is 1.04671


In [ ]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()
    with torch.no_grad():
        for x,y in loader:
            x= x.to(device=device)
            y= y.to(device=device)
            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)
        print(f"accuracy= {float(num_correct) / float(num_samples) * 100:.2f}")

model.train()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
print(f"Accuracy on training set: {check_accuracy(train_loader, model)*100:.2f}")
print(f"Accuracy on test set: {check_accuracy(test_loader, model)*100:.2f}")

accuracy= 65.52


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

In [23]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

    model.train()
    return num_correct / num_samples

In [24]:
print(f"Accuracy on training set: {check_accuracy(train_loader, model)*100:.2f}")
print(f"Accuracy on test set: {check_accuracy(test_loader, model)*100:.2f}")

Accuracy on training set: 65.45
Accuracy on test set: 65.71


### Mobilenet_v2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2  # Cats vs Dogs

# Load MobileNetV2
model = torchvision.models.mobilenet_v2(weights='DEFAULT')

# MobileNetV2 does NOT have avgpool like VGG → remove this line
# model.avgpool = nn.Identity()  # ← WRONG for MobileNetV2

# Replace classifier (1280 features)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)

model = model.to(device)

# Hyperparameters
learning_rate = 3e-4
batch_size = 16
num_epochs = 3

# Data transforms (better suited for MobileNetV2 – use 224x224)
trans_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

trans_valid = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
data_dir = r"D:\Machine Learning\Datasets\Datasets\CatsVsDogs"  # use raw string
train_dataset = torchvision.datasets.ImageFolder(data_dir + '/train', transform=trans_train)
test_dataset  = torchvision.datasets.ImageFolder(data_dir + '/val', transform=trans_valid)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=4)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    loop = tqdm(train_loader, leave=True)
    
    for batch_idx, (data, targets) in enumerate(loop):
        data    = data.to(device)
        targets = targets.to(device)  # long integers
        
        scores = model(data)
        loss   = criterion(scores, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
# Accuracy function (fixed: return float, added model.train() restore)
def check_accuracy(loader, model):
    model.eval()
    num_correct = 0
    num_samples = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += y.size(0)
    model.train()  # restore training mode
    return num_correct / num_samples

# Print accuracies
train_acc = check_accuracy(train_loader, model)
print(f"Accuracy on training set: {train_acc*100:.2f}%")

Epoch [2/3]:  33%|███▎      | 6/18 [00:13<00:22,  1.89s/it, loss=0.506]

### Vgg 16 Model

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

num_classes = 10  # CIFAR-10 has 10 classes

# Load VGG16
model = torchvision.models.vgg16(weights='DEFAULT')

# Freeze all layers (fine-tuning only classifier)
for param in model.parameters():
    param.requires_grad = False

# Replace avgpool and classifier
model.avgpool = nn.Identity()
model.classifier = nn.Sequential(
    nn.Linear(512, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)

model = model.to(device)

# Hyperparameters
learning_rate = 1e-3
batch_size = 64
num_epochs = 3

# Data (CIFAR-10 already has 32x32 images, but VGG expects 224x224 → resize)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.CIFAR10(
    root="dataset/", train=True, download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training
for epoch in range(num_epochs):
    model.train()
    losses = []
    loop = tqdm(train_loader)
    
    for data, targets in loop:
        data    = data.to(device)
        targets = targets.to(device)
        
        scores = model(data)
        loss   = criterion(scores, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
    
    print(f"Cost at epoch {epoch+1}: {sum(losses)/len(losses):.5f}")

# Accuracy function (fixed: return float, added model.train() restore)
def check_accuracy(loader, model):
    model.eval()
    num_correct = 0
    num_samples = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += y.size(0)
    model.train()  # restore training mode
    return num_correct / num_samples

# Print accuracies
train_acc = check_accuracy(train_loader, model)
print(f"Accuracy on training set: {train_acc*100:.2f}%")

# If you have a test_loader, use it here
# test_acc = check_accuracy(test_loader, model)
# print(f"Accuracy on test set: {test_acc*100:.2f}%")

cpu


  0%|          | 0/782 [00:44<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x25088 and 512x100)